# 11 - Boosting multimodal avanzado (XGBoost/CatBoost/LightGBM)

Objetivo: buscar modelos de mayor capacidad con HPO agresiva para maximizar f1-macro en ES, EN y ES+EN.

In [ ]:
import json
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_val_predict
from sklearn.metrics import f1_score, classification_report, make_scorer

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)

In [ ]:
# Si faltan librerias, descomenta e instala en tu entorno de entrenamiento.
# %pip install xgboost lightgbm catboost

import importlib

AVAILABLE_MODELS = {}

def try_register(module_name, class_name, alias):
    try:
        mod = importlib.import_module(module_name)
        cls = getattr(mod, class_name)
        AVAILABLE_MODELS[alias] = cls
    except Exception:
        pass

try_register('xgboost', 'XGBClassifier', 'xgboost')
try_register('lightgbm', 'LGBMClassifier', 'lightgbm')
try_register('catboost', 'CatBoostClassifier', 'catboost')

if len(AVAILABLE_MODELS) == 0:
    raise ImportError('No hay boosting libraries disponibles. Instala xgboost/lightgbm/catboost.')

print('Modelos disponibles:', list(AVAILABLE_MODELS.keys()))

In [ ]:
def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd] + list(cwd.parents)
    for c in candidates:
        if (c / 'materials' / 'dataset_task3_exist2026').exists() and (c / 'notebooks_shiyi').exists():
            return c / 'notebooks_shiyi'
    if (cwd / 'artifacts').exists() and (cwd / 'entregables').exists():
        return cwd
    raise FileNotFoundError('No se pudo resolver notebooks_shiyi.')

PROJECT_ROOT = resolve_project_root()
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
PRED_DIR = PROJECT_ROOT / 'predicciones'
WEIGHTS_DIR = PROJECT_ROOT / 'weights_hpo'
REPORTS_DIR = PROJECT_ROOT / 'reports_hpo'
for d in [PRED_DIR, WEIGHTS_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

bundle = np.load(ARTIFACTS_DIR / 'fusion_features.npz', allow_pickle=True)
X_all = bundle['X_fusion'].astype(np.float32)
y_all = bundle['y'].astype(np.int64)
langs_all = bundle['langs'].astype(str)
sample_ids = bundle['sample_ids'].astype(str)

print('X_all:', X_all.shape, '| etiquetas validas:', int((y_all >= 0).sum()))

In [ ]:
def get_search_space(model_name):
    if model_name == 'xgboost':
        model = AVAILABLE_MODELS['xgboost'](
            objective='binary:logistic',
            eval_metric='logloss',
            random_state=SEED,
            n_jobs=-1
        )
        params = {
            'n_estimators': [300, 600, 900],
            'max_depth': [4, 6, 8, 10],
            'learning_rate': [0.01, 0.03, 0.05, 0.1],
            'subsample': [0.7, 0.85, 1.0],
            'colsample_bytree': [0.6, 0.8, 1.0],
            'reg_alpha': [0.0, 0.1, 1.0],
            'reg_lambda': [1.0, 3.0, 8.0]
        }
        return model, params

    if model_name == 'lightgbm':
        model = AVAILABLE_MODELS['lightgbm'](
            objective='binary',
            class_weight='balanced',
            random_state=SEED,
            n_jobs=-1,
            verbose=-1
        )
        params = {
            'n_estimators': [400, 700, 1000],
            'learning_rate': [0.01, 0.03, 0.05, 0.1],
            'num_leaves': [31, 63, 127],
            'max_depth': [-1, 8, 12],
            'subsample': [0.7, 0.85, 1.0],
            'colsample_bytree': [0.6, 0.8, 1.0],
            'reg_alpha': [0.0, 0.1, 1.0],
            'reg_lambda': [0.0, 1.0, 5.0]
        }
        return model, params

    if model_name == 'catboost':
        model = AVAILABLE_MODELS['catboost'](
            loss_function='Logloss',
            eval_metric='F1',
            random_seed=SEED,
            verbose=False
        )
        params = {
            'iterations': [400, 700, 1000],
            'depth': [4, 6, 8, 10],
            'learning_rate': [0.01, 0.03, 0.05, 0.1],
            'l2_leaf_reg': [1, 3, 5, 9],
            'subsample': [0.7, 0.85, 1.0]
        }
        return model, params

    raise ValueError(model_name)

def tune_threshold(y_true, probs):
    best_t, best_f1 = 0.5, -1.0
    for t in np.linspace(0.20, 0.80, 121):
        pred = (probs >= t).astype(int)
        f1 = f1_score(y_true, pred, average='macro')
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)
    return best_t, best_f1

In [ ]:
def export_json(sample_ids, pred_binary, output_path):
    rows = [
        {'test_case': 'EXIST2026', 'id': str(sid), 'value': 'YES' if int(v) == 1 else 'NO'}
        for sid, v in zip(sample_ids, pred_binary)
    ]
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(rows, f, ensure_ascii=False)
    print('Exportado:', output_path, '| filas:', len(rows))

def run_config(tag, train_langs, n_iter=50):
    mask = np.isin(langs_all, train_langs) & (y_all >= 0)
    X_train = X_all[mask]
    y_train = y_all[mask]

    if len(np.unique(y_train)) < 2:
        raise RuntimeError(f'{tag}: subset sin dos clases.')

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    scorer = make_scorer(f1_score, average='macro')

    best = {'score': -1.0}
    rows = []

    for model_name in AVAILABLE_MODELS.keys():
        estimator, param_dist = get_search_space(model_name)
        rs = RandomizedSearchCV(
            estimator=estimator,
            param_distributions=param_dist,
            n_iter=n_iter,
            scoring=scorer,
            n_jobs=-1,
            cv=cv,
            random_state=SEED,
            verbose=1,
            refit=True
        )
        rs.fit(X_train, y_train)

        probs_oof = cross_val_predict(rs.best_estimator_, X_train, y_train, cv=cv, method='predict_proba', n_jobs=-1)[:, 1]
        best_t, best_cv_thr = tune_threshold(y_train, probs_oof)

        row = {
            'config': tag,
            'model': model_name,
            'best_cv_f1_macro': float(rs.best_score_),
            'best_cv_f1_macro_with_threshold': float(best_cv_thr),
            'best_threshold': float(best_t)
        }
        row.update({f'param_{k}': v for k, v in rs.best_params_.items()})
        rows.append(row)

        if best_cv_thr > best['score']:
            best = {
                'score': float(best_cv_thr),
                'model_name': model_name,
                'estimator': rs.best_estimator_,
                'threshold': float(best_t),
                'base_cv': float(rs.best_score_)
            }

    report_df = pd.DataFrame(rows).sort_values('best_cv_f1_macro_with_threshold', ascending=False)
    report_path = REPORTS_DIR / f'boosting_hpo_{tag}.csv'
    report_df.to_csv(report_path, index=False)
    print('Reporte:', report_path)

    best['estimator'].fit(X_train, y_train)
    probs_all = best['estimator'].predict_proba(X_all)[:, 1]
    pred_all = (probs_all >= best['threshold']).astype(int)

    model_path = WEIGHTS_DIR / f'best_boosting_{tag}_{best['model_name']}.joblib'
    joblib.dump(best, model_path)
    print(f"{tag} => best={best['model_name']} | cv={best['base_cv']:.4f} | cv_thr={best['score']:.4f} | thr={best['threshold']:.3f}")
    print(classification_report(y_train, (best['estimator'].predict_proba(X_train)[:, 1] >= best['threshold']).astype(int), digits=4))

    out_path = PRED_DIR / f'BeingChillingWeWillWin_BOOST_{tag}.json'
    export_json(sample_ids, pred_all, out_path)

run_config('ES', ['es'])
run_config('EN', ['en'])
run_config('ES_EN', ['es', 'en'])